In [5]:
from pathlib import Path

import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

PROJECT_ROOT = Path(r"C:\Case Study Trees\AdultCensusStudy")

SPLIT_DIR = PROJECT_ROOT / "data" / "splits"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

for folder in [
    PROCESSED_DIR / "onehot",
    PROCESSED_DIR / "native",
    ARTIFACT_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

In [6]:
train_df = pd.read_csv(SPLIT_DIR / "train.csv")
valid_df = pd.read_csv(SPLIT_DIR / "valid.csv")
test_df = pd.read_csv(SPLIT_DIR / "test.csv")


y_train = train_df["income"].copy()
y_valid = valid_df["income"].copy()
y_test = test_df["income"].copy()


train_ids = train_df["ROW_ID"].copy()
valid_ids = valid_df["ROW_ID"].copy()
test_ids = test_df["ROW_ID"].copy()


columns_to_drop = ["income", "ROW_ID", "education", "fnlwgt"]

X_train = train_df.drop(columns=columns_to_drop).copy()
X_valid = valid_df.drop(columns=columns_to_drop).copy()
X_test = test_df.drop(columns=columns_to_drop).copy()

assert X_train.columns.equals(X_valid.columns)
assert X_train.columns.equals(X_test.columns)

print("Training:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

Training: (22775, 12)
Validation: (4881, 12)
Test: (4881, 12)


In [7]:
numeric_columns = X_train.select_dtypes(
    include="number"
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    include="object"
).columns.tolist()

print("Numeric predictors:", len(numeric_columns))
print(numeric_columns)

print("\nCategorical predictors:", len(categorical_columns))
print(categorical_columns)

print("\nTraining missing values:")
print(X_train.isna().sum()[X_train.isna().sum() > 0])

Numeric predictors: 5
['age', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']

Categorical predictors: 7
['workclass', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'native.country']

Training missing values:
workclass         1305
occupation        1310
native.country     406
dtype: int64


In [9]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True,
        keep_empty_features=True
    ))
])

onehot_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Missing",
        keep_empty_features=True
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32
    ))
])

native_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Missing",
        keep_empty_features=True
    ))
])

from sklearn.base import clone

onehot_preprocessor = ColumnTransformer([
    ("numeric", clone(numeric_pipeline), numeric_columns),
    (
        "categorical",
        onehot_categorical_pipeline,
        categorical_columns
    )
])

native_preprocessor = ColumnTransformer([
    ("numeric", clone(numeric_pipeline), numeric_columns),
    (
        "categorical",
        native_categorical_pipeline,
        categorical_columns
    )
])

In [10]:
onehot_preprocessor.fit(X_train)
native_preprocessor.fit(X_train)

joblib.dump(
    onehot_preprocessor,
    ARTIFACT_DIR / "adult_onehot_preprocessor.joblib"
)

joblib.dump(
    native_preprocessor,
    ARTIFACT_DIR / "adult_native_preprocessor.joblib"
)

print("Both preprocessors fitted and saved.")

Both preprocessors fitted and saved.
